# Generate tree - coverage plantMASST

In [1]:
import pandas as pd

In [2]:
ncbi_families = pd.read_csv("/Users/helenarusso/Python Scripts/plantMASST/plantMASST_update_nov_2025/ncbi_plant_families_Embryophyta.tsv", sep="\t")

plantmasst = pd.read_csv('/Users/helenarusso/Documents/Pesquisa/UCSD Post-doc/PlantMASST/Submission_Nature/Revision1/updating metadata nov2025/final/plant_masst_table_lineage_tree.tsv', sep="\t")
plantmasst = plantmasst.rename(columns={"TaxID": "tax_id"})
#keep only family populated rows
plantmasst_families = plantmasst[plantmasst["family"].notna()]



In [3]:
ncbi_families

,tax_id,scientific_name,rank,parent_tax_id,division,lineage
0,4185,Acanthaceae,family,4143,Plants and Fungi,cellular organisms; Eukaryota; Viridiplantae; ...
1,112800,Achariaceae,family,3646,Plants and Fungi,cellular organisms; Eukaryota; Viridiplantae; ...
2,25636,Achatocarpaceae,family,3524,Plants and Fungi,cellular organisms; Eukaryota; Viridiplantae; ...
3,42228,Acoraceae,family,91812,Plants and Fungi,cellular organisms; Eukaryota; Viridiplantae; ...
4,118069,Acrobolbaceae,family,3200,Plants and Fungi,cellular organisms; Eukaryota; Viridiplantae; ...
...,...,...,...,...,...,...
729,75435,Xyridaceae,family,38820,Plants and Fungi,cellular organisms; Eukaryota; Viridiplantae; ...
730,3298,Zamiaceae,family,3297,Plants and Fungi,cellular organisms; Eukaryota; Viridiplantae; ...
731,4642,Zingiberaceae,family,4618,Plants and Fungi,cellular organisms; Eukaryota; Viridiplantae; ...
732,27254,Zosteraceae,family,16360,Plants and Fungi,cellular organisms; Eukaryota; Viridiplantae; ...


In [4]:
from ete3 import NCBITaxa

ncbitaxa = NCBITaxa()

# Quick diagnostics
print('Rows in plantmasst_families:', len(plantmasst_families))
print('Unique family names in plantmasst_families:', plantmasst_families['family'].nunique())
print('Unique tax_id in plantmasst_families:', plantmasst_families['tax_id'].nunique())

plant_tax_ids = (
    pd.to_numeric(plantmasst_families['tax_id'], errors='coerce')
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)

rank_lookup = ncbitaxa.get_rank(plant_tax_ids)
rank_counts = pd.Series([rank_lookup.get(t, 'unknown') for t in plant_tax_ids]).value_counts()
print('\nObserved rank distribution for plantmasst_families tax_id:')
print(rank_counts.head(10))

# Convert each PlantMASST tax_id to its family ancestor tax_id
lineages = ncbitaxa.get_lineage_translator(plant_tax_ids)
all_ancestors = sorted({anc for lineage in lineages.values() for anc in lineage})
ancestor_ranks = ncbitaxa.get_rank(all_ancestors)

family_tax_ids_from_plantmasst = set()
for lineage in lineages.values():
    for anc in lineage:
        if ancestor_ranks.get(anc) == 'family':
            family_tax_ids_from_plantmasst.add(anc)
            break

print('\nUnique family-level tax_id recovered from PlantMASST lineage:', len(family_tax_ids_from_plantmasst))

# Name-level sanity check against local ncbi_families table
plant_family_names_series = plantmasst_families['family'].dropna().astype(str).str.strip()
plant_family_names = set(plant_family_names_series.str.lower().unique().tolist())
ncbi_family_names = set(
    ncbi_families['scientific_name'].dropna().astype(str).str.strip().str.lower().unique().tolist()
)
missing_family_names = sorted(plant_family_names - ncbi_family_names)

print('PlantMASST family names not found in ncbi_families scientific_name:', len(missing_family_names))
print(missing_family_names[:30])

ncbi_family_ids_set = set(pd.to_numeric(ncbi_families['tax_id'], errors='coerce').dropna().astype(int).tolist())
outside_ncbi_table = sorted(family_tax_ids_from_plantmasst - ncbi_family_ids_set)
print('Recovered family tax_ids not present in ncbi_families table:', len(outside_ncbi_table))
if outside_ncbi_table:
    print(pd.DataFrame({
        'tax_id': outside_ncbi_table,
        'name': [ncbitaxa.get_taxid_translator([t]).get(t, 'NA') for t in outside_ncbi_table]
    }).head(20))

# Global NCBI check by family name (not restricted to your local ncbi_families table)
name_translator = ncbitaxa.get_name_translator(sorted(plant_family_names_series.unique().tolist()))
resolved_in_ncbi_global = set(name_translator.keys())
unresolved_in_ncbi_global = sorted(set(plant_family_names_series.unique().tolist()) - resolved_in_ncbi_global)

print('\nPlantMASST family names resolved in global NCBI taxonomy:', len(resolved_in_ncbi_global))
print('PlantMASST family names not resolved in global NCBI taxonomy:', len(unresolved_in_ncbi_global))
print(unresolved_in_ncbi_global[:30])

Rows in plantmasst_families: 4015
Unique family names in plantmasst_families: 318
Unique tax_id in plantmasst_families: 4015

Observed rank distribution for plantmasst_families tax_id:
species       3524
genus          386
subspecies      43
varietas        28
unknown         22
no rank          4
forma            3
subgenus         2
family           2
genotype         1
Name: count, dtype: int64

Unique family-level tax_id recovered from PlantMASST lineage: 317
PlantMASST family names not found in ncbi_families scientific_name: 3
['chenopodiaceae', 'hyacinthaceae', 'ovulidae']
Recovered family tax_ids not present in ncbi_families table: 3
    tax_id            name
0    44985   Hyacinthaceae
1   217754        Ovulidae
2  1804623  Chenopodiaceae

PlantMASST family names resolved in global NCBI taxonomy: 318
PlantMASST family names not resolved in global NCBI taxonomy: 0
[]


In [3]:
from ete3 import NCBITaxa

ncbitaxa = NCBITaxa()

# NCBI family IDs from the local reference table used to build the tree
ncbi_family_ids = (
    pd.to_numeric(ncbi_families["tax_id"], errors="coerce")
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)

valid_rank_lookup = ncbitaxa.get_rank(ncbi_family_ids)
valid_ncbi_family_ids = [taxid for taxid in ncbi_family_ids if taxid in valid_rank_lookup]

# Build a topology connecting all NCBI families through shared ancestors
family_tree = ncbitaxa.get_topology(valid_ncbi_family_ids, intermediate_nodes=True)

# --- Map PlantMASST entries to family-level tax_ids ---
# 1) Primary strategy: resolve each PlantMASST tax_id lineage and take its family ancestor
plant_tax_ids = (
    pd.to_numeric(plantmasst_families["tax_id"], errors="coerce")
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)
lineage_lookup = ncbitaxa.get_lineage_translator(plant_tax_ids)

all_ancestors = sorted({anc for lineage in lineage_lookup.values() for anc in lineage})
ancestor_ranks = ncbitaxa.get_rank(all_ancestors)

family_ids_from_lineage = set()
for lineage in lineage_lookup.values():
    for anc in lineage:
        if ancestor_ranks.get(anc) == "family":
            family_ids_from_lineage.add(anc)
            break

# 2) Global NCBI name resolution for PlantMASST family labels (captures all 318 names)
plant_family_names = sorted(
    plantmasst_families["family"].dropna().astype(str).str.strip().unique().tolist()
)
name_to_taxids_global = ncbitaxa.get_name_translator(plant_family_names)
family_ids_from_global_name = {taxid for ids in name_to_taxids_global.values() for taxid in ids}

# Combined PlantMASST family IDs in global NCBI taxonomy
plantmasst_family_ids_global = family_ids_from_lineage | family_ids_from_global_name

# Restrict to local ncbi_families table IDs for tree coverage metrics
plantmasst_family_ids_in_table = plantmasst_family_ids_global & set(valid_ncbi_family_ids)

# Build family-level overlap table on local ncbi_families table
family_overlap = pd.DataFrame({"tax_id": valid_ncbi_family_ids})
family_overlap["in_plantmasst"] = family_overlap["tax_id"].isin(plantmasst_family_ids_in_table)
family_overlap["source_lineage"] = family_overlap["tax_id"].isin(family_ids_from_lineage)
family_overlap["source_global_name"] = family_overlap["tax_id"].isin(family_ids_from_global_name)
family_overlap["name"] = family_overlap["tax_id"].map(
    ncbitaxa.get_taxid_translator(valid_ncbi_family_ids)
)

# Annotate tree nodes
all_tree_taxids = [n.taxid for n in family_tree.traverse() if hasattr(n, "taxid")]
name_lookup = ncbitaxa.get_taxid_translator(all_tree_taxids)
for node in family_tree.traverse():
    taxid = getattr(node, "taxid", None)
    node.add_features(
        sci_name=name_lookup.get(taxid, str(taxid) if taxid is not None else "NA"),
        present_in_plantmasst=(taxid in plantmasst_family_ids_in_table),
        mark="PlantMASST" if taxid in plantmasst_family_ids_in_table else "",
    )

print(f"NCBI family tax_ids in local table (valid): {len(valid_ncbi_family_ids)}")
print(f"PlantMASST unique family names: {plantmasst_families['family'].nunique()}")
print(f"PlantMASST family IDs from lineage: {len(family_ids_from_lineage)}")
print(f"PlantMASST family IDs from global NCBI name translation: {len(family_ids_from_global_name)}")
print(f"PlantMASST families found in global NCBI taxonomy (union): {len(plantmasst_family_ids_global)}")
print(f"PlantMASST families present in local ncbi_families table: {len(plantmasst_family_ids_in_table)}")
print(f"Local-table coverage: {100 * family_overlap['in_plantmasst'].mean():.2f}%")

# Optional text tree view (can be large)

family_overlap.sort_values(["in_plantmasst", "name"], ascending=[False, True]).head(25)

/Users/helenarusso/Python Scripts/.venv/lib/python3.12/site-packages/ete3/treeview/faces.py:159: SyntaxWarning: invalid escape sequence '\_'
  """Base Face object. All Face types (i.e. TextFace, SeqMotifFace,


NCBI family tax_ids in local table (valid): 729
PlantMASST unique family names: 318
PlantMASST family IDs from lineage: 317
PlantMASST family IDs from global NCBI name translation: 318
PlantMASST families found in global NCBI taxonomy (union): 318
PlantMASST families present in local ncbi_families table: 315
Local-table coverage: 43.21%

                                                   /-24579, Menyanthaceae, PlantMASST
                                                  |
                                                  |--16472, Goodeniaceae, PlantMASST
                                                  |
                                                  |--4210, Asteraceae, PlantMASST
                                                  |
                                                  |--4381, Campanulaceae, PlantMASST
                                                  |
                                                  |--57707, Argophyllaceae, PlantMASST
                           

,tax_id,in_plantmasst,source_lineage,source_global_name,name
0,4185,True,True,True,Acanthaceae
1,112800,True,True,True,Achariaceae
2,25636,True,True,True,Achatocarpaceae
3,42228,True,True,True,Acoraceae
5,3623,True,True,True,Actinidiaceae
7,4206,True,True,True,Adoxaceae
10,3542,True,True,True,Aizoaceae
12,4449,True,True,True,Alismataceae
15,56740,True,True,True,Alstroemeriaceae
16,91829,True,True,True,Altingiaceae


In [11]:
# Families in ncbi_families found in PlantMASST (after family-level remapping)
families_mapped = family_overlap[family_overlap["in_plantmasst"]].copy()
families_not_mapped = family_overlap[~family_overlap["in_plantmasst"]].copy()

display(
    families_mapped.sort_values("name").reset_index(drop=True)
)

print(f"Mapped families: {len(families_mapped)}")
print(f"Not mapped families: {len(families_not_mapped)}")

# Optional export files
out_prefix = "/Users/helenarusso/Python Scripts/plantMASST/plantMASST_update_nov_2025/plantMASST_coverage_NCBI/"
families_mapped.to_csv(out_prefix + "families_mapped_in_plantmasst.tsv", sep="\t", index=False)
families_not_mapped.to_csv(out_prefix + "families_not_mapped_in_plantmasst.tsv", sep="\t", index=False)
family_tree.write(format=1, outfile=out_prefix + "ncbi_families_tree.newick")

print("Saved:")
print("- families_mapped_in_plantmasst.tsv")
print("- families_not_mapped_in_plantmasst.tsv")
print("- ncbi_families_tree.newick")

,tax_id,in_plantmasst,source_lineage,source_global_name,name
0,4185,True,True,True,Acanthaceae
1,112800,True,True,True,Achariaceae
2,25636,True,True,True,Achatocarpaceae
3,42228,True,True,True,Acoraceae
4,3623,True,True,True,Actinidiaceae
...,...,...,...,...,...
310,1003242,True,True,True,Ximeniaceae
311,3298,True,True,True,Zamiaceae
312,4642,True,True,True,Zingiberaceae
313,27254,True,True,True,Zosteraceae


Mapped families: 315
Not mapped families: 414
Saved:
- families_mapped_in_plantmasst.tsv
- families_not_mapped_in_plantmasst.tsv
- ncbi_families_tree.newick


In [20]:
# Export iTOL color-strip annotation (tip colors) for the Newick tree
covered_family_ids = set(
    family_overlap.loc[family_overlap["in_plantmasst"], "tax_id"].astype(int).tolist()
)

RED = "#d94a57"   # PlantMASST covered
BLUE = "#4aa3df"  # Not covered

itol_out = "/Users/helenarusso/Python Scripts/plantMASST/plantMASST_update_nov_2025/plantMASST_coverage_NCBI/itol_plantmasst_coverage_colorstrip.txt"

rows = []
for leaf in family_tree.iter_leaves():
    leaf_label = str(leaf.name)  # must match leaf IDs in the exported Newick

    taxid = getattr(leaf, "taxid", None)
    if taxid is None:
        try:
            taxid = int(leaf_label)
        except ValueError:
            taxid = None

    is_covered = (taxid in covered_family_ids) if taxid is not None else False
    color = RED if is_covered else BLUE
    label = "PlantMASST covered" if is_covered else "Not covered"
    rows.append((leaf_label, color, label))

with open(itol_out, "w", encoding="utf-8") as f:
    f.write("DATASET_COLORSTRIP\n")
    f.write("SEPARATOR TAB\n")
    f.write("DATASET_LABEL\tPlantMASST coverage\n")
    f.write("COLOR\t#333333\n")
    f.write("STRIP_WIDTH\t25\n")
    f.write("MARGIN\t5\n")
    f.write("BORDER_WIDTH\t0\n")
    f.write("SHOW_INTERNAL\t0\n")
    f.write("LEGEND_TITLE\tPlantMASST coverage\n")
    f.write("LEGEND_SHAPES\t1\t1\n")
    f.write(f"LEGEND_COLORS\t{RED}\t{BLUE}\n")
    f.write("LEGEND_LABELS\tPlantMASST covered\tNot covered\n")
    f.write("DATA\n")
    for leaf_label, color, label in rows:
        f.write(f"{leaf_label}\t{color}\t{label}\n")

print(f"Saved iTOL annotation file: {itol_out}")
print("Upload this file in iTOL: Datasets -> Color strip -> Add dataset")

Saved iTOL annotation file: /Users/helenarusso/Python Scripts/plantMASST/plantMASST_update_nov_2025/plantMASST_coverage_NCBI/itol_plantmasst_coverage_colorstrip.txt
Upload this file in iTOL: Datasets -> Color strip -> Add dataset


In [25]:
# Export iTOL-ready tree + branch-color annotation with propagated clade states
# NOTE: prefix node labels to avoid iTOL treating numeric internal labels as bootstrap supports.
covered_family_ids = set(
    family_overlap.loc[family_overlap["in_plantmasst"], "tax_id"].astype(int).tolist()
)

RED = "#d94a57"   # PlantMASST covered
BLUE = "#4aa3df"  # Not covered

itol_tree = family_tree.copy(method="deepcopy")
unnamed_counter = 1

for node in itol_tree.traverse("postorder"):
    taxid = getattr(node, "taxid", None)
    if taxid is not None:
        node.name = f"tax_{taxid}"
    elif getattr(node, "name", ""):
        node.name = f"node_{str(node.name).replace(' ', '_')}"
    else:
        node.name = f"node_internal_{unnamed_counter}"
        unnamed_counter += 1

itol_newick_out = "/Users/helenarusso/Python Scripts/plantMASST/plantMASST_update_nov_2025/plantMASST_coverage_NCBI/ncbi_families_tree_itol_named_prefixed.newick"
itol_tree.write(format=1, outfile=itol_newick_out)

# Propagate states from tips to root:
# red  = all descendant leaves covered
# blue = all descendant leaves not covered
# mixed = descendants contain both red and blue
for node in itol_tree.traverse("postorder"):
    if node.is_leaf():
        taxid = getattr(node, "taxid", None)
        node_state = "red" if (taxid in covered_family_ids) else "blue"
        node.add_features(clade_state=node_state)
    else:
        child_states = [getattr(ch, "clade_state", "mixed") for ch in node.children]
        if all(state == "red" for state in child_states):
            node_state = "red"
        elif all(state == "blue" for state in child_states):
            node_state = "blue"
        else:
            node_state = "mixed"
        node.add_features(clade_state=node_state)

itol_branch_out = "/Users/helenarusso/Python Scripts/plantMASST/plantMASST_update_nov_2025/plantMASST_coverage_NCBI/itol_plantmasst_coverage_branch_colors_prefixed.txt"

with open(itol_branch_out, "w", encoding="utf-8") as f:
    f.write("TREE_COLORS\n")
    f.write("SEPARATOR TAB\n")
    f.write("DATA\n")
    for node in itol_tree.traverse("postorder"):
        node_id = str(node.name)
        state = getattr(node, "clade_state", "mixed")
        color = RED if state == "red" else BLUE
        f.write(f"{node_id}\tbranch\t{color}\tnormal\n")

print(f"Saved iTOL-ready prefixed tree: {itol_newick_out}")
print(f"Saved iTOL branch-color file: {itol_branch_out}")
print("In iTOL, upload ncbi_families_tree_itol_named_prefixed.newick, then add itol_plantmasst_coverage_branch_colors_prefixed.txt")

Saved iTOL-ready prefixed tree: /Users/helenarusso/Python Scripts/plantMASST/plantMASST_update_nov_2025/plantMASST_coverage_NCBI/ncbi_families_tree_itol_named_prefixed.newick
Saved iTOL branch-color file: /Users/helenarusso/Python Scripts/plantMASST/plantMASST_update_nov_2025/plantMASST_coverage_NCBI/itol_plantmasst_coverage_branch_colors_prefixed.txt
In iTOL, upload ncbi_families_tree_itol_named_prefixed.newick, then add itol_plantmasst_coverage_branch_colors_prefixed.txt


In [27]:
# Export iTOL tip-label mapping so tax_* IDs display as family names
# Use with the prefixed tree: ncbi_families_tree_itol_named_prefixed.newick

label_out = "/Users/helenarusso/Python Scripts/plantMASST/plantMASST_update_nov_2025/plantMASST_coverage_NCBI/itol_plantmasst_tip_labels_prefixed.txt"

# Build taxid -> scientific name map for leaves only
leaf_taxids = [
    int(leaf.taxid) for leaf in itol_tree.iter_leaves()
    if getattr(leaf, "taxid", None) is not None
]
name_lookup = ncbitaxa.get_taxid_translator(leaf_taxids)

with open(label_out, "w", encoding="utf-8") as f:
    f.write("LABELS\n")
    f.write("SEPARATOR TAB\n")
    f.write("DATA\n")

    for leaf in itol_tree.iter_leaves():
        node_id = str(leaf.name)
        taxid = getattr(leaf, "taxid", None)

        if taxid is not None:
            sci_name = name_lookup.get(int(taxid), f"taxid_{taxid}")
        else:
            sci_name = node_id

        f.write(f"{node_id}\t{sci_name}\n")

print(f"Saved iTOL labels file: {label_out}")
print("In iTOL, upload this via Datasets -> Labels and Text -> Labels")

Saved iTOL labels file: /Users/helenarusso/Python Scripts/plantMASST/plantMASST_update_nov_2025/plantMASST_coverage_NCBI/itol_plantmasst_tip_labels_prefixed.txt
In iTOL, upload this via Datasets -> Labels and Text -> Labels


In [29]:
# Export iTOL label colors (tip names) using the same red/blue coverage logic
# Use with the prefixed tree and LABELS file already generated.

label_color_out = "/Users/helenarusso/Python Scripts/plantMASST/plantMASST_update_nov_2025/plantMASST_coverage_NCBI/itol_plantmasst_tip_label_colors_prefixed.txt"

covered_family_ids = set(
    family_overlap.loc[family_overlap["in_plantmasst"], "tax_id"].astype(int).tolist()
)

RED = "#d94a57"
BLUE = "#4aa3df"

with open(label_color_out, "w", encoding="utf-8") as f:
    f.write("TREE_COLORS\n")
    f.write("SEPARATOR TAB\n")
    f.write("DATA\n")

    for leaf in itol_tree.iter_leaves():
        node_id = str(leaf.name)
        taxid = getattr(leaf, "taxid", None)
        is_covered = (taxid in covered_family_ids) if taxid is not None else False
        color = RED if is_covered else BLUE

        # label style: normal, size 1
        f.write(f"{node_id}\tlabel\t{color}\tnormal\t1\n")

print(f"Saved iTOL label-color file: {label_color_out}")
print("In iTOL, upload via Datasets -> Tree colors -> Add dataset")
print("This colors tip labels red/blue while branch colors stay in the other TREE_COLORS file.")

Saved iTOL label-color file: /Users/helenarusso/Python Scripts/plantMASST/plantMASST_update_nov_2025/plantMASST_coverage_NCBI/itol_plantmasst_tip_label_colors_prefixed.txt
In iTOL, upload via Datasets -> Tree colors -> Add dataset
This colors tip labels red/blue while branch colors stay in the other TREE_COLORS file.


In [33]:
# Export iTOL outer ring (ColorStrip) showing the class of each family tip

class_ring_out = "/Users/helenarusso/Python Scripts/plantMASST/plantMASST_update_nov_2025/plantMASST_coverage_NCBI/itol_family_class_ring_prefixed.txt"

# Collect family tip taxids from the prefixed iTOL tree
leaf_taxids = [
    int(leaf.taxid) for leaf in itol_tree.iter_leaves()
    if getattr(leaf, "taxid", None) is not None
]

lineage_lookup = ncbitaxa.get_lineage_translator(leaf_taxids)
all_ancestors = sorted({anc for lin in lineage_lookup.values() for anc in lin})
ancestor_ranks = ncbitaxa.get_rank(all_ancestors)

# Map family taxid -> class taxid (closest class ancestor in lineage)
family_to_class_taxid = {}
for fam_taxid in leaf_taxids:
    lineage = lineage_lookup.get(fam_taxid, [])
    class_taxid = next(
        (anc for anc in reversed(lineage) if ancestor_ranks.get(anc) == "class"),
        None,
    )
    family_to_class_taxid[fam_taxid] = class_taxid

class_taxids = sorted({t for t in family_to_class_taxid.values() if t is not None})
class_name_lookup = ncbitaxa.get_taxid_translator(class_taxids)

family_to_class_name = {
    fam: class_name_lookup.get(class_taxid, "Unclassified") if class_taxid is not None else "Unclassified"
    for fam, class_taxid in family_to_class_taxid.items()
}

class_names = sorted(set(family_to_class_name.values()))

# Matplotlib Tab20 palette (Tableau 20); cycle if there are >20 classes.
tab20_colors = [
    "#1f77b4", "#aec7e8", "#ff7f0e", "#ffbb78", "#2ca02c",
    "#98df8a", "#d62728", "#ff9896", "#9467bd", "#c5b0d5",
    "#8c564b", "#c49c94", "#e377c2", "#f7b6d2", "#7f7f7f",
    "#c7c7c7", "#bcbd22", "#dbdb8d", "#17becf", "#9edae5",
]
class_to_color = {
    class_name: tab20_colors[i % len(tab20_colors)]
    for i, class_name in enumerate(class_names)
}

legend_shapes = "\t".join(["1"] * len(class_names))
legend_colors = "\t".join([class_to_color[c] for c in class_names])
legend_labels = "\t".join(class_names)

with open(class_ring_out, "w", encoding="utf-8") as f:
    f.write("DATASET_COLORSTRIP\n")
    f.write("SEPARATOR TAB\n")
    f.write("DATASET_LABEL\tFamily class\n")
    f.write("COLOR\t#333333\n")
    f.write("STRIP_WIDTH\t40\n")
    f.write("MARGIN\t8\n")
    f.write("BORDER_WIDTH\t0\n")
    f.write("SHOW_INTERNAL\t0\n")
    f.write("LEGEND_TITLE\tFamily class\n")
    f.write(f"LEGEND_SHAPES\t{legend_shapes}\n")
    f.write(f"LEGEND_COLORS\t{legend_colors}\n")
    f.write(f"LEGEND_LABELS\t{legend_labels}\n")
    f.write("DATA\n")

    for leaf in itol_tree.iter_leaves():
        node_id = str(leaf.name)  # e.g., tax_4209
        taxid = getattr(leaf, "taxid", None)
        class_name = family_to_class_name.get(int(taxid), "Unclassified") if taxid is not None else "Unclassified"
        color = class_to_color[class_name]
        f.write(f"{node_id}\t{color}\t{class_name}\n")

print(f"Saved iTOL class outer-ring file: {class_ring_out}")
print("In iTOL, upload via Datasets -> Color strips -> Add dataset")

Saved iTOL class outer-ring file: /Users/helenarusso/Python Scripts/plantMASST/plantMASST_update_nov_2025/plantMASST_coverage_NCBI/itol_family_class_ring_prefixed.txt
In iTOL, upload via Datasets -> Color strips -> Add dataset


In [34]:
# Export iTOL outer ring (ColorStrip) showing the phylum of each family tip
phylum_ring_out = "/Users/helenarusso/Python Scripts/plantMASST/plantMASST_update_nov_2025/plantMASST_coverage_NCBI/itol_family_phylum_ring_prefixed.txt"

# Collect family tip taxids from the prefixed iTOL tree
leaf_taxids = [
    int(leaf.taxid) for leaf in itol_tree.iter_leaves()
    if getattr(leaf, "taxid", None) is not None
]

lineage_lookup = ncbitaxa.get_lineage_translator(leaf_taxids)
all_ancestors = sorted({anc for lin in lineage_lookup.values() for anc in lin})
ancestor_ranks = ncbitaxa.get_rank(all_ancestors)

# Map family taxid -> phylum taxid (closest phylum ancestor in lineage)
family_to_phylum_taxid = {}
for fam_taxid in leaf_taxids:
    lineage = lineage_lookup.get(fam_taxid, [])
    phylum_taxid = next(
        (anc for anc in reversed(lineage) if ancestor_ranks.get(anc) == "phylum"),
        None,
    )
    family_to_phylum_taxid[fam_taxid] = phylum_taxid

phylum_taxids = sorted({t for t in family_to_phylum_taxid.values() if t is not None})
phylum_name_lookup = ncbitaxa.get_taxid_translator(phylum_taxids)

family_to_phylum_name = {
    fam: phylum_name_lookup.get(phylum_taxid, "Unclassified") if phylum_taxid is not None else "Unclassified"
    for fam, phylum_taxid in family_to_phylum_taxid.items()
}

phylum_names = sorted(set(family_to_phylum_name.values()))

# Matplotlib Tab20 palette (Tableau 20); cycle if there are >20 phyla.
tab20_colors = [
    "#1f77b4", "#aec7e8", "#ff7f0e", "#ffbb78", "#2ca02c",
    "#98df8a", "#d62728", "#ff9896", "#9467bd", "#c5b0d5",
    "#8c564b", "#c49c94", "#e377c2", "#f7b6d2", "#7f7f7f",
    "#c7c7c7", "#bcbd22", "#dbdb8d", "#17becf", "#9edae5",
]
phylum_to_color = {
    phylum_name: tab20_colors[i % len(tab20_colors)]
    for i, phylum_name in enumerate(phylum_names)
}

legend_shapes = "\t".join(["1"] * len(phylum_names))
legend_colors = "\t".join([phylum_to_color[p] for p in phylum_names])
legend_labels = "\t".join(phylum_names)

with open(phylum_ring_out, "w", encoding="utf-8") as f:
    f.write("DATASET_COLORSTRIP\n")
    f.write("SEPARATOR TAB\n")
    f.write("DATASET_LABEL\tFamily phylum\n")
    f.write("COLOR\t#333333\n")
    f.write("STRIP_WIDTH\t40\n")
    f.write("MARGIN\t8\n")
    f.write("BORDER_WIDTH\t0\n")
    f.write("SHOW_INTERNAL\t0\n")
    f.write("LEGEND_TITLE\tFamily phylum\n")
    f.write(f"LEGEND_SHAPES\t{legend_shapes}\n")
    f.write(f"LEGEND_COLORS\t{legend_colors}\n")
    f.write(f"LEGEND_LABELS\t{legend_labels}\n")
    f.write("DATA\n")

    for leaf in itol_tree.iter_leaves():
        node_id = str(leaf.name)  # e.g., tax_4209
        taxid = getattr(leaf, "taxid", None)
        phylum_name = family_to_phylum_name.get(int(taxid), "Unclassified") if taxid is not None else "Unclassified"
        color = phylum_to_color[phylum_name]
        f.write(f"{node_id}\t{color}\t{phylum_name}\n")

print(f"Saved iTOL phylum outer-ring file: {phylum_ring_out}")
print("In iTOL, upload via Datasets -> Color strips -> Add dataset")

Saved iTOL phylum outer-ring file: /Users/helenarusso/Python Scripts/plantMASST/plantMASST_update_nov_2025/plantMASST_coverage_NCBI/itol_family_phylum_ring_prefixed.txt
In iTOL, upload via Datasets -> Color strips -> Add dataset
